In [ ]:
import sqlite3
import json
from datetime import datetime
from flask import Flask, request, jsonify

app = Flask(__name__)
def init_db():
    conn = sqlite3.connect('store.db')
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS webhook_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        received_data TEXT NOT NULL,
        timestamp TEXT NOT NULL
    )
    ''')
    conn.commit()
    conn.close()

init_db()
@app.route('/webhook', methods=['POST'])
def handle_webhook():
    try:
        data = request.get_json()
        print(f" Received Webhook Data: {data}")

        data_str = json.dumps(data)
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        conn = sqlite3.connect('store.db')
        cursor = conn.cursor()
        cursor.execute('''
        INSERT INTO webhook_logs (received_data, timestamp)
        VALUES (?, ?)
        ''', (data_str, current_time))
        conn.commit()
        conn.close()

        return jsonify({"status": "received"}), 200

    except Exception as e:
        print(f"Error handling webhook: {e}")
        return jsonify({"status": "error", "message": str(e)}), 400
if __name__ == '__main__':
    print("Starting Flask Webhook Server Setup...")
    with app.test_client() as client:
        test_payload = {
            "event": "order_created",
            "item": "Dell Laptop",
            "amount": 850
        }

        response = client.post('/webhook', json=test_payload)
        print(f"Response Status Code: {response.status_code}")
        print(f"Response JSON: {response.get_json()}")
    print("\n--- البيانات المخزنة في جدول webhook_logs ---")
    conn = sqlite3.connect('store.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM webhook_logs")
    for row in cursor.fetchall():
        print(f"ID: {row[0]} | Data: {row[1]} | Timestamp: {row[2]}")
    conn.close()